In [0]:
dbutils.widgets.text(
    "batch_id",
    "",
    "Batch ID"
)

dbutils.widgets.dropdown(
    "source_load_type",
    "incremental",
    ["full", "incremental"],
    "Source Load Type"
)

dbutils.widgets.dropdown(
    "execution_mode",
    "standard",
    ["standard", "backfill"],
    "Execution Mode"
)


In [0]:
batch_id = dbutils.widgets.get(
    "batch_id"
)

source_load_type = dbutils.widgets.get(
    "source_load_type"
)

execution_mode = dbutils.widgets.get(
    "execution_mode"
)

if not batch_id:
    raise ValueError(
        "batch_id is required"
    )

print(batch_id)
print(source_load_type)
print(execution_mode)

In [0]:
import uuid
from datetime import (
    datetime,
    timezone
)
from functools import reduce
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window


CATALOG = "github_pipeline_catalog"
SCHEMA = "github_analytics"

LOG_TABLE = (
    f"{CATALOG}.{SCHEMA}."
    "pipeline_execution_logs"
)

In [0]:
def write_log(
    run_id,
    layer,
    dataset,
    batch_id,
    input_parameter,
    source_load_type,
    execution_mode,
    start_time,
    end_time,
    status,
    rows_read,
    rows_inserted,
    rows_updated,
    rows_quarantined,
    error_message=None
):
    log_data = [(
        run_id,
        "github_open_source_analytics",
        layer,
        dataset,
        batch_id,
        input_parameter,
        source_load_type,
        execution_mode,
        start_time,
        end_time,
        status,
        int(rows_read),
        int(rows_inserted),
        int(rows_updated),
        int(rows_quarantined),
        error_message
    )]

    log_schema = """
        run_id STRING,
        pipeline_name STRING,
        layer STRING,
        dataset STRING,
        batch_id STRING,
        input_parameter STRING,
        source_load_type STRING,
        execution_mode STRING,
        execution_start TIMESTAMP,
        execution_end TIMESTAMP,
        status STRING,
        rows_read LONG,
        rows_inserted LONG,
        rows_updated LONG,
        rows_quarantined LONG,
        error_message STRING
    """

    (
        spark.createDataFrame(
            log_data,
            schema=log_schema
        )
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(LOG_TABLE)
    )

In [0]:
def add_record_hash(
    df,
    columns
):

    values = [
        F.coalesce(
            F.col(c).cast("string"),
            F.lit("<NULL>")
        )
        for c in columns
    ]

    return df.withColumn(
        "record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *values
            ),
            256
        )
    )

In [0]:
def merge_gold(
    source_df,
    target_table,
    key_columns
):

    source_df = (
        source_df
        .dropDuplicates(
            key_columns
        )
    )

    existing = (
        spark.table(target_table)
        .select(
            *key_columns,
            "record_hash"
        )
    )

    source_alias = source_df.alias("s")
    target_alias = existing.alias("t")

    join_conditions = [
        F.col(f"s.{key}")
        ==
        F.col(f"t.{key}")
        for key in key_columns
    ]

    join_condition = reduce(
        lambda a, b: a & b,
        join_conditions
    )

    comparison = (
        source_alias
        .join(
            target_alias,
            join_condition,
            "left"
        )
    )

    first_key = key_columns[0]

    rows_inserted = (
        comparison
        .filter(
            F.col(
                f"t.{first_key}"
            ).isNull()
        )
        .count()
    )

    rows_updated = (
        comparison
        .filter(
            F.col(
                f"t.{first_key}"
            ).isNotNull()
            &
            (
                F.col(
                    "s.record_hash"
                )
                !=
                F.col(
                    "t.record_hash"
                )
            )
        )
        .count()
    )

    merge_condition = " AND ".join(
        [
            f"t.`{key}` = s.`{key}`"
            for key in key_columns
        ]
    )

    target = DeltaTable.forName(
        spark,
        target_table
    )

    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            merge_condition
        )
        .whenMatchedUpdateAll(
            condition=(
                "t.record_hash "
                "<> s.record_hash"
            )
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    return (
        rows_inserted,
        rows_updated
    )

In [0]:
def process_dim_repository():

    dataset = "dim_repository"

    run_id = str(uuid.uuid4())
    start_time = datetime.now(
        timezone.utc
    )

    source = spark.table(
        f"{CATALOG}.{SCHEMA}."
        "silver_repositories"
    )

    rows_read = source.count()

    target_table = (
        f"{CATALOG}.{SCHEMA}."
        "gold_dim_repository"
    )

    try:

        gold = source.select(
            "repo_id",

            F.col("name")
            .alias(
                "repository_name"
            ),

            "full_name",
            "owner_login",
            "language",
            "default_branch",
            "created_at",
            "archived",
            "fork",
            "source_batch_id"
        )

        business_columns = [
            "repo_id",
            "repository_name",
            "full_name",
            "owner_login",
            "language",
            "default_branch",
            "created_at",
            "archived",
            "fork"
        ]

        gold = add_record_hash(
            gold,
            business_columns
        )

        gold = gold.withColumn(
            "load_timestamp",
            F.current_timestamp()
        )

        inserted, updated = (
            merge_gold(
                gold,
                target_table,
                ["repo_id"]
            )
        )

        write_log(
            run_id,
            "Silver-to-Gold",
            dataset,
            batch_id,
            batch_id,
            source_load_type,
            execution_mode,
            start_time,
            datetime.now(
                timezone.utc
            ),
            "SUCCESS",
            rows_read,
            inserted,
            updated,
            0,
            None
        )

        print(
            "dim_repository: SUCCESS | "
            f"inserted={inserted}, "
            f"updated={updated}"
        )

    except Exception as error:

        write_log(
            run_id,
            "Silver-to-Gold",
            dataset,
            batch_id,
            batch_id,
            source_load_type,
            execution_mode,
            start_time,
            datetime.now(
                timezone.utc
            ),
            "FAILURE",
            rows_read,
            0,
            0,
            0,
            str(error)
        )

        raise

In [0]:
def process_fact_repository_snapshot():

    dataset = "fact_repository_snapshot"

    run_id = str(uuid.uuid4())

    start_time = datetime.now(
        timezone.utc
    )

    source = spark.table(
        f"{CATALOG}.{SCHEMA}."
        "silver_repository_snapshots"
    )

    rows_read = source.count()

    target_table = (
        f"{CATALOG}.{SCHEMA}."
        "gold_fact_repository_snapshot"
    )

    try:

        daily_window = (
            Window
            .partitionBy(
                "repo_id",
                "snapshot_date"
            )
            .orderBy(
                F.col(
                    "snapshot_timestamp"
                ).desc(),
                F.col(
                    "load_timestamp"
                ).desc()
            )
        )

        daily = (
            source
            .withColumn(
                "daily_rank",
                F.row_number()
                .over(daily_window)
            )
            .filter(
                F.col("daily_rank")
                == 1
            )
            .drop("daily_rank")
        )

        history_window = (
            Window
            .partitionBy("repo_id")
            .orderBy("snapshot_date")
        )

        fact = (
            daily

            .withColumn(
                "previous_stars",
                F.lag("stars")
                .over(history_window)
            )

            .withColumn(
                "previous_forks",
                F.lag("forks")
                .over(history_window)
            )

            .withColumn(
                "previous_open_issues",
                F.lag("open_issues")
                .over(history_window)
            )

            .withColumn(
                "star_change",
                F.coalesce(
                    F.col("stars")
                    -
                    F.col(
                        "previous_stars"
                    ),
                    F.lit(0)
                )
            )

            .withColumn(
                "fork_change",
                F.coalesce(
                    F.col("forks")
                    -
                    F.col(
                        "previous_forks"
                    ),
                    F.lit(0)
                )
            )

            .withColumn(
                "open_issue_change",
                F.coalesce(
                    F.col("open_issues")
                    -
                    F.col(
                        "previous_open_issues"
                    ),
                    F.lit(0)
                )
            )

            .withColumn(
                "date_key",
                F.date_format(
                    "snapshot_date",
                    "yyyyMMdd"
                ).cast("int")
            )

            .select(
                "repo_id",
                "date_key",
                "stars",
                "forks",
                "open_issues",
                "star_change",
                "fork_change",
                "open_issue_change",
                F.col("batch_id")
                .alias(
                    "source_batch_id"
                )
            )
        )

        business_columns = (
            fact.columns
        )

        fact = add_record_hash(
            fact,
            business_columns
        )

        fact = fact.withColumn(
            "load_timestamp",
            F.current_timestamp()
        )

        inserted, updated = merge_gold(
            fact,
            target_table,
            [
                "repo_id",
                "date_key"
            ]
        )

        write_log(
            run_id,
            "Silver-to-Gold",
            dataset,
            batch_id,
            batch_id,
            source_load_type,
            execution_mode,
            start_time,
            datetime.now(
                timezone.utc
            ),
            "SUCCESS",
            rows_read,
            inserted,
            updated,
            0,
            None
        )

        print(
            "fact_repository_snapshot: "
            "SUCCESS | "
            f"inserted={inserted}, "
            f"updated={updated}"
        )

    except Exception as error:

        write_log(
            run_id,
            "Silver-to-Gold",
            dataset,
            batch_id,
            batch_id,
            source_load_type,
            execution_mode,
            start_time,
            datetime.now(
                timezone.utc
            ),
            "FAILURE",
            rows_read,
            0,
            0,
            0,
            str(error)
        )

        raise

In [0]:
def process_fact_repository_activity():

    dataset = "fact_repository_activity"

    run_id = str(uuid.uuid4())

    start_time = datetime.now(
        timezone.utc
    )

    commits = spark.table(
        f"{CATALOG}.{SCHEMA}."
        "silver_commits"
    )

    issues = spark.table(
        f"{CATALOG}.{SCHEMA}."
        "silver_issues"
    )

    prs = spark.table(
        f"{CATALOG}.{SCHEMA}."
        "silver_pull_requests"
    )

    rows_read = (
        commits.count()
        +
        issues.count()
        +
        prs.count()
    )

    target_table = (
        f"{CATALOG}.{SCHEMA}."
        "gold_fact_repository_activity"
    )

    try:

        commits_daily = (
            commits
            .filter(
                F.col(
                    "author_date"
                ).isNotNull()
            )
            .withColumn(
                "activity_date",
                F.to_date(
                    "author_date"
                )
            )
            .groupBy(
                "repo_id",
                "activity_date"
            )
            .agg(
                F.countDistinct(
                    "commit_sha"
                ).alias(
                    "commit_count"
                ),

                F.countDistinct(
                    "author_login_hash"
                ).alias(
                    "active_commit_authors"
                )
            )
        )

        issues_opened = (
            issues
            .filter(
                F.col(
                    "created_at"
                ).isNotNull()
            )
            .withColumn(
                "activity_date",
                F.to_date(
                    "created_at"
                )
            )
            .groupBy(
                "repo_id",
                "activity_date"
            )
            .agg(
                F.countDistinct(
                    "issue_id"
                ).alias(
                    "issues_opened"
                )
            )
        )

        issues_closed = (
            issues
            .filter(
                F.col(
                    "closed_at"
                ).isNotNull()
                &
                F.col(
                    "created_at"
                ).isNotNull()
            )
            .withColumn(
                "activity_date",
                F.to_date(
                    "closed_at"
                )
            )
            .withColumn(
                "resolution_hours",
                (
                    F.unix_timestamp(
                        "closed_at"
                    )
                    -
                    F.unix_timestamp(
                        "created_at"
                    )
                ) / 3600.0
            )
            .groupBy(
                "repo_id",
                "activity_date"
            )
            .agg(
                F.countDistinct(
                    "issue_id"
                ).alias(
                    "issues_closed"
                ),

                F.sum(
                    "resolution_hours"
                ).alias(
                    "issue_resolution_hours_sum"
                )
            )
        )

        prs_created = (
            prs
            .filter(
                F.col(
                    "created_at"
                ).isNotNull()
            )
            .withColumn(
                "activity_date",
                F.to_date(
                    "created_at"
                )
            )
            .groupBy(
                "repo_id",
                "activity_date"
            )
            .agg(
                F.countDistinct(
                    "pr_id"
                ).alias(
                    "pull_requests_created"
                )
            )
        )

        prs_merged = (
            prs
            .filter(
                F.col(
                    "merged_at"
                ).isNotNull()
                &
                F.col(
                    "created_at"
                ).isNotNull()
            )
            .withColumn(
                "activity_date",
                F.to_date(
                    "merged_at"
                )
            )
            .withColumn(
                "merge_hours",
                (
                    F.unix_timestamp(
                        "merged_at"
                    )
                    -
                    F.unix_timestamp(
                        "created_at"
                    )
                ) / 3600.0
            )
            .groupBy(
                "repo_id",
                "activity_date"
            )
            .agg(
                F.countDistinct(
                    "pr_id"
                ).alias(
                    "pull_requests_merged"
                ),

                F.sum(
                    "merge_hours"
                ).alias(
                    "pr_merge_hours_sum"
                )
            )
        )

        metric_frames = [
            commits_daily,
            issues_opened,
            issues_closed,
            prs_created,
            prs_merged
        ]

        activity = reduce(
            lambda left, right:
                left.join(
                    right,
                    [
                        "repo_id",
                        "activity_date"
                    ],
                    "full_outer"
                ),
            metric_frames
        )

        activity = activity.fillna(
            {
                "commit_count": 0,
                "active_commit_authors": 0,
                "issues_opened": 0,
                "issues_closed": 0,
                "issue_resolution_hours_sum": 0.0,
                "pull_requests_created": 0,
                "pull_requests_merged": 0,
                "pr_merge_hours_sum": 0.0
            }
        )

        activity = (
            activity
            .withColumn(
                "date_key",
                F.date_format(
                    "activity_date",
                    "yyyyMMdd"
                ).cast("int")
            )
            .select(
                "repo_id",
                "date_key",
                "commit_count",
                "active_commit_authors",
                "issues_opened",
                "issues_closed",
                "issue_resolution_hours_sum",
                "pull_requests_created",
                "pull_requests_merged",
                "pr_merge_hours_sum"
            )
        )

        business_columns = (
            activity.columns
        )

        activity = add_record_hash(
            activity,
            business_columns
        )

        activity = (
            activity
            .withColumn(
                "refresh_batch_id",
                F.lit(batch_id)
            )
            .withColumn(
                "load_timestamp",
                F.current_timestamp()
            )
        )

        inserted, updated = merge_gold(
            activity,
            target_table,
            [
                "repo_id",
                "date_key"
            ]
        )

        write_log(
            run_id,
            "Silver-to-Gold",
            dataset,
            batch_id,
            batch_id,
            source_load_type,
            execution_mode,
            start_time,
            datetime.now(
                timezone.utc
            ),
            "SUCCESS",
            rows_read,
            inserted,
            updated,
            0,
            None
        )

        print(
            "fact_repository_activity: "
            "SUCCESS | "
            f"inserted={inserted}, "
            f"updated={updated}"
        )

    except Exception as error:

        write_log(
            run_id,
            "Silver-to-Gold",
            dataset,
            batch_id,
            batch_id,
            source_load_type,
            execution_mode,
            start_time,
            datetime.now(
                timezone.utc
            ),
            "FAILURE",
            rows_read,
            0,
            0,
            0,
            str(error)
        )

        raise

In [0]:
def process_fact_pipeline_execution():

    dataset = "fact_pipeline_execution"

    run_id = str(uuid.uuid4())

    start_time = datetime.now(
        timezone.utc
    )

    source = spark.table(
        f"{CATALOG}.{SCHEMA}."
        "pipeline_execution_logs"
    )

    rows_read = source.count()

    target_table = (
        f"{CATALOG}.{SCHEMA}."
        "gold_fact_pipeline_execution"
    )

    try:

        fact = (
            source

            .withColumn(
                "date_key",
                F.date_format(
                    "execution_start",
                    "yyyyMMdd"
                ).cast("int")
            )

            .withColumn(
                "execution_duration_seconds",
                (
                    F.unix_timestamp(
                        "execution_end"
                    )
                    -
                    F.unix_timestamp(
                        "execution_start"
                    )
                ).cast("long")
            )

            .select(
                "run_id",
                "date_key",
                "layer",
                "dataset",
                "status",
                "source_load_type",
                "execution_mode",
                "rows_read",
                "rows_inserted",
                "rows_updated",
                "rows_quarantined",
                "execution_duration_seconds",
                "error_message"
            )
        )

        business_columns = (
            fact.columns
        )

        fact = add_record_hash(
            fact,
            business_columns
        )

        fact = fact.withColumn(
            "load_timestamp",
            F.current_timestamp()
        )

        inserted, updated = merge_gold(
            fact,
            target_table,
            ["run_id"]
        )

        write_log(
            run_id,
            "Silver-to-Gold",
            dataset,
            batch_id,
            batch_id,
            source_load_type,
            execution_mode,
            start_time,
            datetime.now(
                timezone.utc
            ),
            "SUCCESS",
            rows_read,
            inserted,
            updated,
            0,
            None
        )

        print(
            "fact_pipeline_execution: "
            "SUCCESS | "
            f"inserted={inserted}, "
            f"updated={updated}"
        )

    except Exception as error:

        write_log(
            run_id,
            "Silver-to-Gold",
            dataset,
            batch_id,
            batch_id,
            source_load_type,
            execution_mode,
            start_time,
            datetime.now(
                timezone.utc
            ),
            "FAILURE",
            rows_read,
            0,
            0,
            0,
            str(error)
        )

        raise

In [0]:
print(
    "Starting Silver -> Gold"
)

process_dim_repository()
process_fact_repository_snapshot()
process_fact_repository_activity()
process_fact_pipeline_execution()

print(
    "Silver -> Gold completed"
)